Test to see API Access

In [2]:
from dotenv import load_dotenv
import os

load_dotenv()
key = os.environ.get("DATABENTO_API_KEY", "NOT SET")

if key == "NOT SET":
    print("Still NOT SET. Check that .env exists and is named correctly.")
else:
    print(f"Key loaded. First 6 chars: {key[:6]}, last 4 chars: {key[-4:]}")
    print(f"Full length: {len(key)}")

Key loaded. First 6 chars: db-r6f, last 4 chars: H5CE
Full length: 32


In [20]:
import pandas as pd
from pathlib import Path


Smoke Test

In [4]:
from dotenv import load_dotenv
import os
import databento as db

load_dotenv()

client = db.Historical(os.environ["DATABENTO_API_KEY"])

cost = client.metadata.get_cost(
    dataset="OPRA.PILLAR",
    symbols=["SPX.OPT"],
    stype_in="parent",
    schema="cmbp-1",
    start="2026-06-17T13:30:00",
    end="2026-06-17T14:30:00",
)
print(f"Estimated cost: ${cost:.4f}")

Estimated cost: $2.2540


In [13]:
# Get ALL instrument definitions for one day
# This returns metadata about every options contract, not quote data,
# so it's small.
definitions = client.timeseries.get_range(
    dataset="OPRA.PILLAR",
    schema="definition",
    symbols="ALL_SYMBOLS",
    start="2025-06-17",
    end="2025-06-18",
)
defs_df = definitions.to_df()
print(f"Total contracts on 2025-06-17: {len(defs_df):,}")

# Filter to SPX only
spx_defs = defs_df[defs_df['underlying'] == 'SPX']
print(f"SPX contracts: {len(spx_defs):,}")

if len(spx_defs) > 0:
    print(spx_defs[['raw_symbol', 'instrument_class', 'strike_price',
                    'expiration', 'underlying']].head(10))
    print(f"\nUnique expirations (first 20):")
    print(sorted(spx_defs['expiration'].unique())[:20])

Total contracts on 2025-06-17: 1,629,537
SPX contracts: 8,266
                                                raw_symbol instrument_class  \
ts_recv                                                                       
2025-06-17 10:30:00.477669916+00:00  SPX   261218C05900000                C   
2025-06-17 10:30:00.477669916+00:00  SPX   261218P07200000                P   
2025-06-17 10:30:00.478105630+00:00  SPX   261218C02000000                C   
2025-06-17 10:30:00.478110530+00:00  SPX   261218C02400000                C   
2025-06-17 10:30:00.488831890+00:00  SPX   251219P05700000                P   
2025-06-17 10:30:00.488831890+00:00  SPX   251219P04800000                P   
2025-06-17 10:30:00.491122645+00:00  SPX   251219P04500000                P   
2025-06-17 10:30:00.491122645+00:00  SPX   251219C04000000                C   
2025-06-17 10:30:00.493617525+00:00  SPX   261218C06000000                C   
2025-06-17 10:30:00.493617525+00:00  SPX   261218C03200000           

In [16]:
# Pick strikes near 6000 (approximate ATM) for a few expiries spanning TTE
# Weekly: 2025-06-20 (3 days out)
# Monthly: 2025-07-18 (~1 month out)  
# Quarterly: 2025-09-19 (~3 months out)

target_expiries = [
    pd.Timestamp('2025-06-20', tz='UTC'),
    pd.Timestamp('2025-07-18', tz='UTC'),
    pd.Timestamp('2025-09-19', tz='UTC'),
]
target_strikes = [5900.0, 6000.0, 6100.0]

# Filter definitions to our basket
basket = spx_defs[
    (spx_defs['expiration'].isin(target_expiries)) &
    (spx_defs['strike_price'].isin(target_strikes))
].copy()

print(f"Basket size: {len(basket)}")
print(basket[['raw_symbol', 'instrument_class', 'strike_price', 'expiration']].to_string())

Basket size: 18
                                                raw_symbol instrument_class  strike_price                expiration
ts_recv                                                                                                            
2025-06-17 10:30:01.033747351+00:00  SPX   250620C05900000                C        5900.0 2025-06-20 00:00:00+00:00
2025-06-17 10:30:01.234115373+00:00  SPX   250718C05900000                C        5900.0 2025-07-18 00:00:00+00:00
2025-06-17 10:30:06.960440386+00:00  SPX   250620C06000000                C        6000.0 2025-06-20 00:00:00+00:00
2025-06-17 10:30:07.506298319+00:00  SPX   250718C06100000                C        6100.0 2025-07-18 00:00:00+00:00
2025-06-17 10:30:07.575257875+00:00  SPX   250718P06100000                P        6100.0 2025-07-18 00:00:00+00:00
2025-06-17 10:30:07.595165186+00:00  SPX   250718P05900000                P        5900.0 2025-07-18 00:00:00+00:00
2025-06-17 10:30:13.715681240+00:00  SPX   250620C061000

In [17]:
# Extract the raw symbols for our basket
basket_symbols = basket['raw_symbol'].tolist()
print(f"Requesting {len(basket_symbols)} symbols")

# Cost estimate first
cost = client.metadata.get_cost(
    dataset="OPRA.PILLAR",
    symbols=basket_symbols,
    stype_in="raw_symbol",
    schema="cmbp-1",
    start="2025-06-17T13:30:00",  # 9:30 AM ET
    end="2025-06-17T14:30:00",    # 10:30 AM ET
)
print(f"Estimated cost for 1 hour of quotes on 18 contracts: ${cost:.4f}")

Requesting 18 symbols
Estimated cost for 1 hour of quotes on 18 contracts: $0.0079


In [18]:
# Actual pull
quotes = client.timeseries.get_range(
    dataset="OPRA.PILLAR",
    symbols=basket_symbols,
    stype_in="raw_symbol",
    schema="cmbp-1",
    start="2025-06-17T13:30:00",
    end="2025-06-17T14:30:00",
)
quotes_df = quotes.to_df()
print(f"Rows: {len(quotes_df):,}")
print(f"Columns: {quotes_df.columns.tolist()}")
print(f"\nFirst few rows:")
quotes_df.head(10)

Rows: 663,591
Columns: ['ts_event', 'rtype', 'publisher_id', 'instrument_id', 'action', 'side', 'price', 'size', 'flags', 'ts_in_delta', 'bid_px_00', 'ask_px_00', 'bid_sz_00', 'ask_sz_00', 'bid_pb_00', 'ask_pb_00', 'symbol']

First few rows:


,ts_event,rtype,publisher_id,instrument_id,action,side,price,size,flags,ts_in_delta,bid_px_00,ask_px_00,bid_sz_00,ask_sz_00,bid_pb_00,ask_pb_00,symbol
ts_recv,,,,,,,,,,,,,,,,,
2025-06-17 13:30:02.079213554+00:00,2025-06-17 13:30:02.079010890+00:00,177,30,1275073581,A,N,42.6,5,194,0,41.4,42.6,5,5,22,22,SPX 250620C06000000
2025-06-17 13:30:02.079515545+00:00,2025-06-17 13:30:02.079311648+00:00,177,30,1275083715,A,N,32.5,1,194,0,31.8,32.5,20,1,22,22,SPX 250620P06000000
2025-06-17 13:30:02.108836409+00:00,2025-06-17 13:30:02.108632053+00:00,177,30,1275089099,A,N,96.6,5,194,0,91.0,96.6,5,5,22,22,SPX 250620P06100000
2025-06-17 13:30:02.109878754+00:00,2025-06-17 13:30:02.109674281+00:00,177,30,1275078978,A,N,5.0,10,194,0,4.6,5.0,80,10,22,22,SPX 250620C06100000
2025-06-17 13:30:02.110047828+00:00,2025-06-17 13:30:02.109845134+00:00,177,30,1275084015,A,N,10.8,44,194,0,10.3,10.8,20,44,22,22,SPX 250620P05900000
2025-06-17 13:30:02.110610436+00:00,2025-06-17 13:30:02.110406978+00:00,177,30,1275068847,A,N,125.4,27,194,0,117.2,125.4,5,27,22,22,SPX 250620C05900000
2025-06-17 13:30:02.114911635+00:00,2025-06-17 13:30:02.114708450+00:00,177,30,1258315060,A,N,238.5,25,194,0,236.9,238.5,25,25,22,22,SPX 250919C06000000
2025-06-17 13:30:02.116596296+00:00,2025-06-17 13:30:02.116392525+00:00,177,22,1258320942,T,N,175.0,1,194,0,NaN,NaN,0,0,0,0,SPX 250919P06000000
2025-06-17 13:30:02.116742552+00:00,2025-06-17 13:30:02.116537805+00:00,177,30,1258320942,A,N,176.0,12,194,0,174.9,176.0,11,12,22,22,SPX 250919P06000000


In [19]:
# 1. Which contracts appear? Should be all 18.
print("Contracts in quote data:")
print(quotes_df['symbol'].value_counts())

# 2. What actions do we see?
print("\nAction distribution:")
print(quotes_df['action'].value_counts())

# 3. Any weird price artifacts (zeros, negatives)?
print("\nPrice stats:")
print(quotes_df['price'].describe())

# 4. Bid-ask spread stats (only where we have both)
mask = (quotes_df['bid_px_00'].notna()) & (quotes_df['ask_px_00'].notna())
spreads = quotes_df.loc[mask, 'ask_px_00'] - quotes_df.loc[mask, 'bid_px_00']
print("\nSpread stats:")
print(spreads.describe())

Contracts in quote data:
symbol
SPX   250620P06000000    67058
SPX   250620P05900000    56694
SPX   250620C06000000    50461
SPX   250718P06000000    48039
SPX   250718P05900000    45902
SPX   250718C06100000    45632
SPX   250718C06000000    41782
SPX   250919C05900000    39382
SPX   250620C06100000    38637
SPX   250919C06000000    34907
SPX   250919P06000000    34488
SPX   250919P06100000    30872
SPX   250919C06100000    30240
SPX   250919P05900000    29179
SPX   250718C05900000    26832
SPX   250718P06100000    21041
SPX   250620C05900000    12741
SPX   250620P06100000     9704
Name: count, dtype: int64

Action distribution:
action
A    663152
T       439
Name: count, dtype: int64

Price stats:
count    663591.000000
mean        112.187539
std          85.934544
min           4.750000
25%          41.900000
50%          94.100000
75%         175.400000
max         329.500000
Name: price, dtype: float64

Spread stats:
count    663588.000000
mean          1.101042
std           1.38

In [21]:
Path("data/raw").mkdir(parents=True, exist_ok=True)

quotes_df.to_parquet("data/raw/spx_smoke_2025-06-17_quotes.parquet")

basket[['raw_symbol', 'instrument_class', 'strike_price', 'expiration',
        'instrument_id']].to_parquet("data/raw/spx_smoke_2025-06-17_basket.parquet")

for f in Path("data/raw").glob("spx_smoke*"):
    size_mb = os.path.getsize(f) / (1024**2)
    print(f"{f.name}: {size_mb:.2f} MB")

spx_smoke_2025-06-17_quotes.parquet: 13.75 MB
spx_smoke_2025-06-17_basket.parquet: 0.00 MB


In [22]:
# Pull definitions for one specific reference day to see contracts
ref_date = "2025-06-16"  # a Monday
next_date = "2025-06-17"

definitions = client.timeseries.get_range(
    dataset="OPRA.PILLAR",
    schema="definition",
    symbols="ALL_SYMBOLS",
    start=ref_date,
    end=next_date,
)
defs_df = definitions.to_df()
spx_defs = defs_df[defs_df['underlying'] == 'SPX'].copy()

# Filter to reasonable TTE buckets from the ref_date
import pandas as pd
ref_ts = pd.Timestamp(ref_date, tz='UTC')
spx_defs['dte'] = (spx_defs['expiration'] - ref_ts).dt.days

# Pick expiries in each bucket, roughly ATM strikes
# SPX was around 6000 in June 2025
atm = 6000
strike_range = (atm * 0.95, atm * 1.05)  # ATM ± 5%

filtered = spx_defs[
    (spx_defs['strike_price'] >= strike_range[0]) &
    (spx_defs['strike_price'] <= strike_range[1]) &
    (spx_defs['dte'] >= 0) &
    (spx_defs['dte'] <= 200)  # within 200 days
].copy()

print(f"Total contracts in basket: {len(filtered)}")
print(f"\nBy DTE bucket:")
filtered['bucket'] = pd.cut(filtered['dte'], 
                              bins=[-1, 7, 30, 90, 200],
                              labels=['weekly', 'monthly', 'quarterly', 'semi-annual'])
print(filtered.groupby('bucket', observed=True).size())

# Get the raw symbols
basket_symbols = filtered['raw_symbol'].tolist()

Total contracts in basket: 1202

By DTE bucket:
bucket
weekly         226
quarterly      422
semi-annual    554
dtype: int64


In [23]:
# Narrower basket: tighter ATM range, specific expiries
atm = 6000
strike_offsets = [-120, -60, 0, 60, 120]  # ATM ± 2%, ATM ± 1%, ATM (in $ terms)
target_strikes = [atm + off for off in strike_offsets]

# For each TTE bucket, we want to select a small number of specific expiries
# Weekly: nearest 2 weekly expiries
# Monthly: nearest 1 standard monthly (3rd Friday)
# Quarterly: nearest 1 quarterly
# Semi-annual: nearest 1 semi-annual expiry

filtered = spx_defs[
    (spx_defs['strike_price'].isin(target_strikes)) &
    (spx_defs['dte'] >= 0) &
    (spx_defs['dte'] <= 200)
].copy()

# See what expiries exist for these strikes
print(f"Total narrowed basket: {len(filtered)}")
print(f"\nUnique expiries in basket:")
print(sorted(filtered['expiration'].dt.date.unique()))

print(f"\nDTE distribution:")
print(filtered.groupby(filtered['dte']).size().to_string())

Total narrowed basket: 62

Unique expiries in basket:
[datetime.date(2025, 6, 20), datetime.date(2025, 7, 18), datetime.date(2025, 8, 15), datetime.date(2025, 9, 19), datetime.date(2025, 10, 17), datetime.date(2025, 11, 21), datetime.date(2025, 12, 19)]

DTE distribution:
dte
4      10
32     10
60     10
95     10
123    10
158    10
186     2


In [24]:
basket_symbols = filtered['raw_symbol'].tolist()
print(f"Number of symbols: {len(basket_symbols)}")

# One full trading week: Mon 2025-06-16 to Fri 2025-06-20
# But 2025-06-20 was Juneteenth (US market holiday), so use 06-16 to 06-19
# Actually let's use a clean week: 2025-06-23 to 2025-06-27
week_cost = client.metadata.get_cost(
    dataset="OPRA.PILLAR",
    symbols=basket_symbols,
    stype_in="raw_symbol",
    schema="cmbp-1",
    start="2025-06-23T13:30:00",  # Monday 9:30 AM ET
    end="2025-06-27T20:00:00",    # Friday 4:00 PM ET (SPX trades until 4:00 ET)
)
print(f"Estimated cost for 1 week of 62 contracts: ${week_cost:.4f}")

# Extrapolate to 52 weeks
print(f"Extrapolated to 12 months: ${week_cost * 52:.2f}")

Number of symbols: 62
Estimated cost for 1 week of 62 contracts: $0.4533
Extrapolated to 12 months: $23.57


In [ ]:
import numpy as np
data = np.load("data/sequences/spx_smoke_2025-06-17_h50/sequences.npz")
print("Files:", data.files)
print("sequences shape:", data["sequences"].shape, "dtype:", data["sequences"].dtype)
print("labels shape:", data["labels"].shape, "unique:", np.unique(data["labels"]))
print("tte_days shape:", data["tte_days"].shape, "range:", data["tte_days"].min(), "-", data["tte_days"].max())
print("contract_ids shape:", data["contract_ids"].shape, "unique:", np.unique(data["contract_ids"]))

# Peek at one sequence
print("\nSample sequence (first 3 timesteps of first sample):")
print(data["sequences"][0, :3, :])